# Model Evaluation on the Ice Cream Dataset

This notebook evaluates multiple trained architectures on the same structured test dataset that describes human traits and contextual factors influencing ice cream buying behavior.  
Each model predicts a scalar probability $\hat{y} \in [0, 1]$, representing the likelihood of buying ice cream, as estimated by a reference language model.

---

## 1. Objective

The goal of this evaluation is to assess and compare the predictive quality of various neural and non-neural architectures trained on the Ice Cream Dataset.

Specifically, the evaluation measures:
- **Mean Squared Error (MSE):** Quantifies the average squared difference between predicted and target probabilities.
- **Coefficient of Determination (R²):** Measures how much of the variance in the target data can be explained by the model.

Both metrics are computed over the full test dataset to ensure a fair and comprehensive comparison.

---

## 2. Dataset

The test set is provided in JSONL format (`test.jsonl`).  
Each entry contains:
- a list of **traits** (e.g. `"likes sweets"`, `"health-conscious"`),
- a list of **contextual factors** (e.g. `"hot summer day"`, `"after lunch"`),
- and a scalar **target probability** `probability_LLM`.

These inputs are tokenized into integer IDs using the shared vocabulary from the training stage.

Two data representations are used:
- **Tensor-based input** for PyTorch models, processed via a custom `IceCreamDataset` and `DataLoader`.
- **Binary bag-of-words representation** for tree-based models (Decision Tree, Random Forest).

---

## 3. Evaluated Architectures

### Neural Models (PyTorch)
1. **Logistic Regression Model** — a single-layer linear model over averaged embeddings.  
2. **Feedforward MLP** — multiple fully-connected layers with nonlinear activations.  
3. **Encoder-only Transformer** — parallel self-attention over tokens without positional encoding.  
4. **Decoder-only Transformer** — autoregressive masked attention structure.

### Tree-based Models (scikit-learn)
5. **Decision Tree Regressor** — recursive partitioning of the input feature space into piecewise constant regions.  
6. **Random Forest Regressor** — ensemble of decision trees trained with random data and feature sampling.

---

## 4. Evaluation Metrics

For each model, predictions are computed on the **entire test dataset**:

$$
\text{MSE} = \frac{1}{N} \sum_i (y_i - \hat{y}_i)^2
$$

$$
R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}
$$

Where:
- $y_i$ vare the true target probabilities from the LLM,
- $\hat{y}_i$ are the model predictions,
- $\bar{y}$ is the mean of all true targets.

A lower MSE and higher R² indicate better model performance.

---

## 5. Procedure

1. **Load all saved models** from the directory `../models/saved_models/`.  
2. **Load test data** via `IceCreamDataset` (for PyTorch) and `load_tree_data` (for sklearn).  
3. **Run evaluation** using dedicated functions for each model type.  
4. **Collect metrics** (MSE, R²) in a single table for comparison.

In [4]:
# EVALUATE MODELS ON ICE CREAM DATASET

import torch
import torch.nn as nn
import joblib
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
import json
import os
import pandas as pd
import sys
sys.path.append("../") 
from src.models.basic_models import (
    LogisticRegressionModel,
    FeedForwardMLP,
    BasicEncoderTransformer,
    BasicDecoderTransformer
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# LOAD DATA


vocab = [
    "likes sweets", "dislikes sweets", "health-conscious", "lactose intolerant",
    "cheap", "spender", "impulsive buyer",
    "hungry", "on a diet", "ice cream truck nearby", "hot summer day",
    "cold winter day", "ice cream is cheap today (discount)",
    "after a long workout", "after lunch", "after work"
]

token2id = {tok: i for i, tok in enumerate(vocab)}

# Für PyTorch-Modelle
class IceCreamDataset(torch.utils.data.Dataset):
    def __init__(self, path):
        self.data = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                self.data.append(json.loads(line.strip()))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]
        ids = [token2id[tok] for tok in ex["traits"] + ex["context"]]
        x = torch.tensor(ids, dtype=torch.long)
        y = torch.tensor(ex["probability_LLM"], dtype=torch.float32)
        return x, y

def collate_fn(batch):
    input_ids = [b[0] for b in batch]
    targets = torch.stack([b[1] for b in batch])
    max_len = max(len(x) for x in input_ids)
    padded = torch.stack([torch.cat([x, torch.zeros(max_len - len(x), dtype=torch.long)]) for x in input_ids])
    return padded, targets

eval_dataset = IceCreamDataset("../data/test.jsonl")
eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=8, collate_fn=collate_fn)


# LOAD TRAINED MODELS

# Hilfsfunktion für PyTorch-Modelle
def load_torch_model(model_class, path, vocab_size):
    model = model_class(vocab_size=vocab_size)
    model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    model.eval().to(device)
    return model

# Lade alle Modelle
torch_models = {
    "Logistic Regression": load_torch_model(LogisticRegressionModel, "../models/saved_models/LogisticRegressionModel_d128_lr0.001_epoch10.pt", len(vocab)),
    "Feedforward MLP": load_torch_model(FeedForwardMLP, "../models/saved_models/FeedForwardMLP_d128_lr0.001_epoch10.pt", len(vocab)),
    "Encoder Transformer": load_torch_model(BasicEncoderTransformer, "../models/saved_models/BasicEncoderTransformer_d128_lr0.0001_epoch10.pt", len(vocab)),
    "Decoder Transformer": load_torch_model(BasicDecoderTransformer, "../models/saved_models/BasicDecoderTransformer_d128_lr0.0001_epoch10.pt", len(vocab))
}

# Sklearn-Modelle laden
sklearn_models = {
    "Decision Tree": joblib.load("../models/saved_models/DecisionTree_depth5.pkl"),
    "Random Forest": joblib.load("../models/saved_models/RandomForest_100x6.pkl")
}

# EVALUATION FUNCTIONS

def evaluate_torch_model(model, dataloader):
    preds, targets = [], []
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            p = model(x)
            preds.extend(p.cpu().numpy())
            targets.extend(y.cpu().numpy())
    preds = np.array(preds)
    targets = np.array(targets)
    return mean_squared_error(targets, preds), r2_score(targets, preds)

def load_tree_data(path, vocab):
    import numpy as np
    data, targets = [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line.strip())
            vec = np.zeros(len(vocab))
            for tok in ex["traits"] + ex["context"]:
                if tok in vocab:
                    vec[vocab.index(tok)] = 1.0
            data.append(vec)
            targets.append(ex["probability_LLM"])
    return np.array(data), np.array(targets)

def evaluate_sklearn_model(model, path, vocab):
    X, y = load_tree_data(path, vocab)
    preds = model.predict(X)
    return mean_squared_error(y, preds), r2_score(y, preds)



Using device: cuda


In [5]:
# RUN EVALUATION FOR ALL MODELS

results = {}

# Torch models
for name, model in torch_models.items():
    mse, r2 = evaluate_torch_model(model, eval_loader)
    results[name] = {"MSE": mse, "R2": r2}
    print(f"{name:<25} | MSE={mse:.4f} | R2={r2:.4f}")

# Sklearn models
for name, model in sklearn_models.items():
    mse, r2 = evaluate_sklearn_model(model, "../data/test.jsonl", vocab)
    results[name] = {"MSE": mse, "R2": r2}
    print(f"{name:<25} | MSE={mse:.4f} | R2={r2:.4f}")

df_results = pd.DataFrame(results).T
df_results = df_results.sort_values(by="MSE")
display(df_results)

Logistic Regression       | MSE=0.0115 | R2=0.8093
Feedforward MLP           | MSE=0.0067 | R2=0.8891
Encoder Transformer       | MSE=0.0062 | R2=0.8965
Decoder Transformer       | MSE=0.0032 | R2=0.9472
Decision Tree             | MSE=0.0122 | R2=0.7984
Random Forest             | MSE=0.0076 | R2=0.8748


,MSE,R2
Decoder Transformer,0.003187,0.947197
Encoder Transformer,0.006247,0.896510
Feedforward MLP,0.006697,0.889065
Random Forest,0.007560,0.874755
Logistic Regression,0.011513,0.809278
Decision Tree,0.012168,0.798419


In [7]:
save_dir = "../results/metrics"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "eval_results.csv")

# Extract ground-truth data and meta info per test sample
records = []

for idx, ex in enumerate(eval_dataset.data):
    n_traits = len(ex["traits"])
    n_context = len(ex["context"])
    y_true = ex["probability_LLM"]

    row = {
        "sample_id": idx,
        "num_traits": n_traits,
        "num_context": n_context,
        "total_tokens": n_traits + n_context,
        "y_true": y_true,
    }
    records.append(row)

df_detailed = pd.DataFrame(records)

# Collect predictions from all models

for name, model in torch_models.items():
    preds = []
    with torch.no_grad():
        for x, _ in eval_loader:
            x = x.to(device)
            p = model(x).cpu().numpy().tolist()
            preds.extend(p)
    df_detailed[f"{name}_pred"] = preds
    df_detailed[f"{name}_sq_error"] = (df_detailed["y_true"] - df_detailed[f"{name}_pred"])**2

from sklearn.metrics import mean_squared_error
def get_sklearn_preds(model, path, vocab):
    X, y = load_tree_data(path, vocab)
    return model.predict(X).tolist()

for name, model in sklearn_models.items():
    preds = get_sklearn_preds(model, "../data/test.jsonl", vocab)
    df_detailed[f"{name}_pred"] = preds
    df_detailed[f"{name}_sq_error"] = (df_detailed["y_true"] - df_detailed[f"{name}_pred"])**2

df_detailed.to_csv(save_path, index=False)
print(f"✅ Detailed evaluation results saved to {save_path}")
display(df_detailed.head())

✅ Detailed evaluation results saved to ../results/metrics/eval_results.csv


,sample_id,num_traits,num_context,total_tokens,y_true,Logistic Regression_pred,Logistic Regression_sq_error,Feedforward MLP_pred,Feedforward MLP_sq_error,Encoder Transformer_pred,Encoder Transformer_sq_error,Decoder Transformer_pred,Decoder Transformer_sq_error,Decision Tree_pred,Decision Tree_sq_error,Random Forest_pred,Random Forest_sq_error
0,0,2,2,4,0.30,0.386773,0.007530,0.288613,0.000130,0.284149,0.000251,0.258478,0.001724,0.437147,0.018809,0.299234,5.867504e-07
1,1,4,2,6,0.35,0.458586,0.011791,0.564649,0.046074,0.476290,0.015949,0.577779,0.051883,0.641439,0.084937,0.606271,6.567479e-02
2,2,5,1,6,0.15,0.191935,0.001759,0.227256,0.005969,0.179575,0.000875,0.165148,0.000229,0.293130,0.020486,0.260397,1.218750e-02
3,3,2,5,7,0.85,0.928287,0.006129,0.912238,0.003874,0.891181,0.001696,0.884898,0.001218,0.903502,0.002862,0.856131,3.759073e-05
4,4,2,5,7,0.30,0.281970,0.000325,0.321115,0.000446,0.320869,0.000436,0.296275,0.000014,0.283427,0.000275,0.293840,3.794069e-05
